# 11 · LangGraph agent + Bedrock Responses API

How to plug an RaitBox sandbox into a **framework** agent
(LangGraph) as a tool, using a different Bedrock invocation path.

**Key concepts**
- **LangGraph `create_react_agent`** — a prebuilt ReAct loop (reason → act → observe)
  that manages tool calls for you; you only supply the model and the tools.
- **Bedrock Responses API** — the OpenAI-compatible endpoint on `bedrock-runtime`
  (`.../openai/v1`). Here `gpt-oss-120b` is driven through `langchain-openai`'s
  `ChatOpenAI(use_responses_api=True)` pointed at Bedrock (auth via a Bedrock API key).
- **Framework-agnostic tool** — the same "run code in a sandbox" tool works across
  frameworks; only the glue differs (compare with `03`'s raw Converse loop).

**How it works:** the tool wraps `sandbox.run_code`; LangGraph routes model tool
calls to it and feeds results back automatically.

**Prerequisites:** `pip install` extras (in-cell); a Bedrock API key
(`OPENAI_API_KEY`) and model access.

In [ ]:
%pip install -q raitbox langgraph langchain-openai

### Configuration
Point `langchain-openai`'s `ChatOpenAI` (Responses-API mode) at the Bedrock OpenAI-compatible endpoint; auth is a **Bedrock API key**.

In [ ]:
import os

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

from raitbox import Sandbox

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-west-2")
MODEL_ID = os.environ.get("RAITBOX_BEDROCK_MODEL", "us.openai.gpt-oss-120b")
BASE_URL = os.environ.get("OPENAI_BASE_URL", f"https://bedrock-runtime.{REGION}.amazonaws.com/openai/v1")

llm = ChatOpenAI(
    model=MODEL_ID, base_url=BASE_URL, api_key=os.environ["OPENAI_API_KEY"],
    use_responses_api=True, temperature=0,
)

### Sandbox as a tool
Wrap `sandbox.run_code` as a LangChain `@tool` the agent can call.

In [ ]:
class CodeInterpreter:
    def __init__(self):
        self.sandbox = Sandbox.create(region_name=REGION)

    def run_code(self, code):
        execution = self.sandbox.run_code(code)
        return {
            "results": [r.text for r in execution.results],
            "stdout": execution.logs.stdout,
            "stderr": execution.logs.stderr,
            "error": None if execution.error is None else execution.error.name,
        }

    def close(self):
        self.sandbox.kill()


interpreter = CodeInterpreter()


@tool
def code_interpreter(code: str) -> dict:
    """Execute Python in a secure sandbox; returns results, stdout, stderr, error."""
    return interpreter.run_code(code)

### Build & run the agent
LangGraph's prebuilt ReAct agent routes the model's tool calls to the sandbox and loops until it produces an answer.

In [ ]:
agent = create_react_agent(
    model=llm,
    tools=[code_interpreter],
    prompt="You are a precise assistant. For anything computational, WRITE and RUN "
    "Python via the code_interpreter tool and answer from the actual output.",
)
try:
    result = agent.invoke({"messages": [{"role": "user",
        "content": "Compute the 25th Fibonacci number and the sum of primes below 1000."}]})
    print(result["messages"][-1].content)
finally:
    interpreter.close()

## Recap & where to go next
You used a real agent framework with an RaitBox tool and the Bedrock Responses API.

**Related:** `03` (raw Converse tool loop) · `02`/`06` (hand-rolled loops).